# Mamba-3 `siso-187m` smoke on free Colab (T4)

Runtime → Change runtime type → **T4 GPU**.

This checks the **training loop** on CUDA with the pure-PyTorch SISO port (`--backend torch`).
Official `mamba_ssm` kernels are optional and often painful on free Colab; skip them for this smoke.

Expect ~10–30 minutes the first time (checkpoint download + a few train steps). Stop the runtime when finished.

In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
assert torch.cuda.is_available(), "Enable a T4 GPU: Runtime → Change runtime type → T4"

In [ ]:
# Clone the branch that has mamba3.train (boss_ranker).
import os
BRANCH = "boss_ranker"
REPO = "https://github.com/CornellDataScience/adhd.git"
if not os.path.isdir("adhd"):
    !git clone --branch {BRANCH} --depth 1 {REPO} adhd
%cd adhd
!git rev-parse --short HEAD
!ls mamba3/train | head

In [ ]:
# Use Colab's torch. Install only what the trainer needs.
!pip -q install -U "einops>=0.8" "huggingface_hub>=0.30" "transformers>=4.50"
!pip -q install -e . --no-deps
from mamba3.train.smoke import main as _
print("import ok")

## Smoke: 5 steps on `siso-187m`

T4 has no bfloat16. The smoke CLI will use fp16 autocast on CUDA.
Seq length 64 keeps the free-tier memory budget safe for a first pass.

In [ ]:
!mkdir -p /content/smoke-187m
!python -m mamba3.train.smoke \
  --model siso-187m \
  --backend torch \
  --device cuda \
  --dtype float16 \
  --steps 5 \
  --seq-len 64 \
  --batch-size 1 \
  --out /content/smoke-187m \
  --log /content/smoke-187m/smoke.jsonl

In [ ]:
import json
from pathlib import Path
log = Path("/content/smoke-187m/smoke.jsonl")
rows = [json.loads(line) for line in log.read_text().splitlines() if line.strip()]
print("steps", len(rows))
print("first loss", rows[0]["loss"])
print("last loss", rows[-1]["loss"])
print("checkpoint", list(Path("/content/smoke-187m/final").iterdir()))
assert (Path("/content/smoke-187m/final") / "pytorch_model.bin").exists()
print("SMOKE OK")

## Optional: longer seq / more steps

Only if the cell above succeeded and you still have GPU time. Free Colab can reclaim the T4.

In [ ]:
# Uncomment to push harder on the same runtime:
# !python -m mamba3.train.smoke --model siso-187m --backend torch --device cuda --dtype float16 \
#   --steps 10 --seq-len 256 --batch-size 1 --out /content/smoke-187m-256

## After Colab

1. **Runtime → Disconnect and delete runtime** so you are not sitting on a GPU.
2. If this smoke passed, the next paid Runpod step is the same command with `--backend auto` after `uv sync --extra cuda`.
3. Do **not** start a multi-day CPT on free Colab.